In [36]:
import pandas as pd
import numpy as np

try:
    df_job = pd.read_csv('한국직업정보 재직자조사 전처리 완료.csv')
    df_uni = pd.read_csv('전국대학별학과정보.csv')
    df_job.columns = df_job.columns.str.strip()
    df_uni.columns = df_uni.columns.str.strip()

    print("두 파일 모두 성공적으로 불러왔습니다!\n")
    print(f"재직자 데이터 크기: {df_job.shape[0]:,}행, {df_job.shape[1]}개 컬럼")
    print(f"대학 학과 데이터 크기: {df_uni.shape[0]:,}행, {df_uni.shape[1]}개 컬럼\n")
    print("=== 재직자 데이터 결측치 확인 ===")
    print(df_job[['직업명', '성취/노력', '인내', '책임성과 진취성']].isnull().sum())

    print("\n=== 재직자 데이터 샘플 ===")
    display(df_job[['직업명', '성취/노력', '인내', '책임성과 진취성']].head(3))

    print("\n=== 대학 학과 데이터 샘플 ===")
    display(df_uni[['학교명', '학과명', '관련직업명']].head(3))

except FileNotFoundError as e:
    print("[에러] 파일명을 찾을 수 없습니다.")
    print("에러 내용:", e)

except Exception as e:
    print("[에러] 데이터를 읽어오는 중 문제가 발생했습니다.")
    print("에러 내용:", e)

/tmp/ipykernel_4811/1472734249.py:5: DtypeWarning: Columns (146) have mixed types. Specify dtype option on import or set low_memory=False.
  df_job = pd.read_csv('한국직업정보 재직자조사 전처리 완료.csv')


두 파일 모두 성공적으로 불러왔습니다!

재직자 데이터 크기: 17,143행, 150개 컬럼
대학 학과 데이터 크기: 60,619행, 23개 컬럼

=== 재직자 데이터 결측치 확인 ===
직업명         0
성취/노력       0
인내          0
책임성과 진취성    0
dtype: int64

=== 재직자 데이터 샘플 ===


,직업명,성취/노력,인내,책임성과 진취성
0,행정부고위공무원,5,5,5
1,행정부고위공무원,5,5,5
2,행정부고위공무원,5,4,5



=== 대학 학과 데이터 샘플 ===


,학교명,학과명,관련직업명
0,ICT폴리텍대학,AI네트워크학과,공무원+네트워크엔지니어+클라우드시스템엔지니어+통신 및 관련 장비설치 및 수리원+통신...
1,ICT폴리텍대학,AI소프트웨어학과,디지털영상처리전문가+방송제작관리자+정보시스템운영자+통신 및 관련 장비설치 및 수리원
2,ICT폴리텍대학,AI영상보안학과,디지털영상처리전문가+방송제작관리자+정보시스템운영자+통신 및 관련 장비설치 및 수리원


In [37]:
interest_features = ['리더십', '혁신']
aptitude_features = ['인내']
value_features = ['성취/노력', '책임성과 진취성', '타인에대한 배려']
features_to_extract = (interest_features + aptitude_features + value_features)

df_filtered = (df_job[['직업명'] + features_to_extract].dropna())

df_job_profile = (df_filtered.groupby('직업명')[features_to_extract].mean().reset_index())
df_job_profile[features_to_extract] = (df_job_profile[features_to_extract].round(2))

print(f"압축 완료. 총 {len(df_job_profile)}개의 고유 직업 프로필이 생성되었습니다.")
print("\n=== 직업별 진로 특성 프로필 샘플 ===\n")
display(df_job_profile.head(5))

df_job_profile.to_csv('직업별_진로특성_프로필.csv', index=False, encoding='utf-8-sig')

print(
    "\n'직업별_진로특성_프로필.csv' "
    "파일 저장 완료!")

압축 완료. 총 570개의 고유 직업 프로필이 생성되었습니다.

=== 직업별 진로 특성 프로필 샘플 ===



,직업명,리더십,혁신,인내,성취/노력,책임성과 진취성,타인에대한 배려
0,3D 프린팅모델러,3.29,3.68,3.71,3.87,3.94,3.55
1,IT 기술지원 전문가,2.73,2.57,3.20,3.37,3.43,2.87
2,IT 테스터 및 IT QA 전문가,3.73,3.30,3.73,3.73,4.23,3.70
3,UX/UI 디자이너,3.47,3.77,3.43,3.63,3.77,3.63
4,가구 제조·수리원,2.87,2.77,3.40,3.53,3.57,3.13



'직업별_진로특성_프로필.csv' 파일 저장 완료!


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_my_career(user_scores, top_n=5):
    """
    사용자 성향 데이터를 기반으로
    직업을 추천하는 함수
    순서:
    [성취/노력, 인내, 책임성과 진취성,
     리더십, 혁신, 타인에대한 배려]
    """

    # 사용자 입력 검증
    if len(user_scores) != len(features_to_extract):

        print("[에러] 성향 점수 개수가 맞지 않습니다.")
        return

    # 직업 특성 벡터 추출
    job_features = (df_job_profile[features_to_extract].values)
    # 사용자 벡터 변환
    user_vector = (np.array(user_scores).reshape(1, -1))
    # 코사인 유사도 계산
    similarity_scores = (cosine_similarity(user_vector, job_features).flatten())
    # 상위 추천 직업 추출
    top_indices = (similarity_scores.argsort()[::-1][:top_n])

    print("=" * 60)
    print("[AI 기반 맞춤형 직업 추천 결과]")
    print("=" * 60)
    print(f"\n입력된 성향 점수: {user_scores}")
    print(
        "(성취/노력, 인내, 책임성과 진취성, "
        "리더십, 혁신, 배려 순)\n")

    # 추천 결과 출력
    for rank, idx in enumerate(top_indices, start=1):
        # 직업명
        job_name = ( df_job_profile.iloc[idx]['직업명'])
        # 유사도 점수
        similarity_score = round(similarity_scores[idx] * 100, 2)
        print(f"{rank}위 추천 직업: {job_name}")
        print(
            f"직업 적합도: "
            f"{similarity_score}%")

        # 사용자 강점 추출
        top_feature_idx = np.argmax(user_scores)
        strong_trait = (features_to_extract[top_feature_idx])
        print(
            f"추천 이유: "
            f"'{strong_trait}' 성향이 "
            f"높은 사용자와 유사합니다.\n")

In [39]:
def integrated_career_recommender(user_scores, top_n=3):

    """
    사용자의 가치관 점수를 기반으로
    직업 + 관련 대학/학과를 추천하는 함수
    """

    # 사용자 입력 검증
    if len(user_scores) != len(features_to_extract):
        print("[에러] 입력된 가치관 점수 개수가 올바르지 않습니다.")
        return

    # 직업 벡터
    job_features = df_job_profile[features_to_extract].values
    # 사용자 벡터
    user_vector = np.array(user_scores).reshape(1, -1)
    # 유사도 계산
    similarity_scores = cosine_similarity(user_vector, job_features).flatten()
    # 상위 추천 직업
    top_indices = similarity_scores.argsort()[::-1][:top_n]
    print("=" * 60)
    print("[최종 진로 및 대학 추천 리포트]")
    print("=" * 60)

    for rank, idx in enumerate(top_indices, start=1):
        job_name = df_job_profile.iloc[idx]['직업명']
        similarity_score = round(similarity_scores[idx] * 100, 2)
        print(f"\n[{rank}위 추천 직업]")
        print(f"직업명: {job_name}")
        print(f"직업 적합도: {similarity_score}%")

        # 관련 학과 검색
        matched_schools = df_uni[df_uni['관련직업명'].astype(str).str.contains(job_name, na=False)]
        # 중복 제거
        unique_schools = (matched_schools[['시도명', '학교명', '학과명']].drop_duplicates())
        # 최대 4개만 출력
        unique_schools = unique_schools.head(4)
        # 결과 출력
        if not unique_schools.empty:
            print("추천 학과 및 대학:")
            for _, row in unique_schools.iterrows():
               print(
                    f"- [{row['시도명']}] "
                    f"{row['학교명']} "
                    f"{row['학과명']}")
        else:
            print(
                "연결 가능한 대학/학과 정보가 "
                "현재 데이터에 없습니다.")

# 테스트용 입력값
final_user_input = [4.0, 3.5, 4.5, 4.5, 4.0, 2.0]
integrated_career_recommender(final_user_input, top_n=3)

[최종 진로 및 대학 추천 리포트]

[1위 추천 직업]
직업명: 웹방송전문가(1인미디어콘텐츠제작자)
직업 적합도: 98.93%
연결 가능한 대학/학과 정보가 현재 데이터에 없습니다.

[2위 추천 직업]
직업명: 데이터베이스개발자
직업 적합도: 98.69%
추천 학과 및 대학:
- [경기도] 가천대학교 컴퓨터공학과
- [경기도] 가천대학교 게임대학원 게임공학과
- [경기도] 가천대학교 일반대학원 AI·소프트웨어학과
- [경기도] 가천대학교 일반대학원 IT융합공학과

[3위 추천 직업]
직업명: 신문기자
직업 적합도: 98.68%
추천 학과 및 대학:
- [경기도] 가천대학교 경제학과
- [경기도] 가천대학교 동양어문학과
- [경기도] 가천대학교 미디어커뮤니케이션학과
- [경기도] 가천대학교 언론영상광고학과


/tmp/ipykernel_4811/1279878684.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matched_schools = df_uni[df_uni['관련직업명'].astype(str).str.contains(job_name, na=False)]


In [40]:
features_to_extract = [
    '성취/노력',
    '인내',
    '책임성과 진취성',
    '리더십',
    '혁신',
    '타인에대한 배려']

def get_web_career_recommendation(user_scores, top_n=3):
    """
    웹 UI용 진로/대학 추천 API 함수

    :param user_scores:
    [성취, 인내, 도전성, 리더십, 혁신, 배려]

    :return:
    웹 화면 출력용 추천 결과 딕셔너리
    """

    # 입력 검증
    if len(user_scores) != len(features_to_extract):
        return {"error": "입력된 가치관 점수 개수가 올바르지 않습니다."}
    # 직업 특성 벡터
    job_features = df_job_profile[features_to_extract].values
    # 사용자 벡터 변환
    user_vector = np.array(user_scores).reshape(1, -1)
    # 유사도 계산
    similarity_scores = cosine_similarity(user_vector, job_features).flatten()
    # 상위 추천 직업 인덱스
    top_indices = similarity_scores.argsort()[::-1][:top_n]
    final_output = []
    for rank, idx in enumerate(top_indices, start=1):
        job_name = df_job_profile.iloc[idx]['직업명']
        similarity_score = round(similarity_scores[idx] * 100, 2)

        # 관련 대학/학과 검색
        matched_schools = df_uni[df_uni['관련직업명'].astype(str).str.contains(job_name, na=False)]
        school_list = []
        if not matched_schools.empty:
            unique_schools = (
                matched_schools[
                    ['시도명', '학교명', '학과명']
                ]
                .drop_duplicates()
                .head(4)
            )

            for _, row in unique_schools.iterrows():
                school_list.append({
                    "region": row['시도명'],
                    "school_name": row['학교명'],
                    "department": row['학과명']
                })

        # 추천 이유 생성
        top_feature_idx = np.argmax(user_scores)
        strong_trait = features_to_extract[top_feature_idx]
        final_output.append({
            "rank": rank,
            "job_name": job_name,
            "similarity_score": f"{similarity_score}%",
            "strong_trait": strong_trait,
            "recommend_reason":
                f"{strong_trait} 성향이 높은 사용자와 "
                f"유사한 직업입니다.",
            "schools": school_list})

    return final_output